In [ ]:
#pip install ydata_profiling

import pandas as pd
import matplotlib as plt
from ydata_profiling import ProfileReport
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
df = pd.read_csv(
    "olist_orders_dataset.csv",
    sep=",",
    on_bad_lines="skip"  # ignora linhas ruins
)

In [ ]:
df.describe()

df.info()

In [ ]:
profile = ProfileReport(df, title="Profiling Report")
profile.to_notebook_iframe()

In [ ]:
#Pré Processamento

cols_datas = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

In [ ]:
for col in cols_datas:
    df[col] = pd.to_datetime(df[col])
	
# criar target de 0 e 1 se teve ou não atraso
# 1 = atraso
# 0 = prazo
df['atraso'] = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int)
df['atraso'].value_counts()

df['tempo_entrega'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
print(df["tempo_entrega"])
df['tempo_estimado'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.days
df['diff_estimado_real'] = df['tempo_entrega'] - df['tempo_estimado']
contagem_entrega = df['tempo_entrega'].isna()
contagem_estimado = df['tempo_estimado'].isna()
contagem_entrega.value_counts()

In [ ]:
contagem_estimado.value_counts()

In [ ]:
# há valores nulos precisamos tratar

df = df.dropna(subset=[
    'tempo_entrega',
    'tempo_estimado',
    'diff_estimado_real',
    'atraso'
])

In [ ]:
#Features

features = ['tempo_entrega', 'tempo_estimado', 'diff_estimado_real']

X = df[features]
y = df['atraso']

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
#Split 80/20

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = Sequential()

In [ ]:
# Camada de entrada + oculta
model.add(Dense(16, activation='relu', input_shape=(X.shape[1],)))

# Segunda camada oculta
model.add(Dense(8, activation='relu'))

# Camada de saída (classificação binária)
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

#Treino do Modelo
history = model.fit(
    X_train, y_train,
    epochs=6,
    validation_data=(X_test, y_test)
)

In [ ]:
#Gráfico para Overfiting

plt.figure()

plt.plot(history.history['loss'], label='Loss Treino')
plt.plot(history.history['val_loss'], label='Loss Validação')

if 'accuracy' in history.history:
    plt.plot(history.history['accuracy'], '--', label='Accuracy Treino')
    plt.plot(history.history['val_accuracy'], '--', label='Accuracy Validação')

plt.title('Comportamento do Modelo durante o Treinamento')
plt.xlabel('Épocas')
plt.ylabel('Métricas')

plt.legend()
plt.grid()

plt.show()

In [ ]:
#Avalialçao

loss, acc = model.evaluate(X_test, y_test)

print("Acurácia:", acc)

In [ ]:
#Verificação de possibilidade de Overfiting
import numpy as np

train_loss = history.history['loss']
val_loss = history.history['val_loss']

# diferença média entre validação e treino
diff = np.mean(np.array(val_loss) - np.array(train_loss))

print("Diferença média (val - treino):", diff)

if diff > 0.05:
    print("Indício de overfitting")
else:
    print("Sem overfitting significativo")

In [ ]:
#Trocando para Relu

model_sigmoid = Sequential()

model_sigmoid.add(Dense(16, activation='sigmoid', input_shape=(X.shape[1],)))
model_sigmoid.add(Dense(8, activation='sigmoid'))
model_sigmoid.add(Dense(1, activation='sigmoid'))

model_sigmoid.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_sigmoid = model_sigmoid.fit(
    X_train, y_train,
    epochs=7,
    validation_data=(X_test, y_test),
    verbose=0
)

loss, acc = model.evaluate(X_test, y_test)

print("Acurácia:", acc)

In [ ]:
#Comparando

print("ReLU final val_loss:", history.history['val_loss'][-1])
print("Sigmoid final val_loss:", history_sigmoid.history['val_loss'][-1])

In [ ]:
#Comparando com um modelo de Regressão Linear

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred)

print("Acurácia Regressão Logística:", acc_lr)

## Conclusao 

### 1. O modelo apresentou overfitting? Como você identificou isso?
Não apresentou overfitting, pois as curvas de erro de treino e validação permaneceram próximas ao longo das épocas, indicando boa generalização.

### 2. A Função ReLU contribui para o aprendizado da rede? Por quê?
Sim. A ReLU contribui ao acelerar o treinamento e evitar o problema de gradientes muito pequenos, permitindo um aprendizado mais eficiente.

### 3. Acurácia alta significa que o modelo é confiável neste contexto?
Não necessariamente. A acurácia deve ser analisada com cautela, pois pode ser influenciada por desbalanceamento ou por variáveis fortemente correlacionadas com o alvo.

### 4. Esse problema poderia ser resolvido com um modelo mais simples? Explique.
Sim. Como as variáveis possuem relação direta com o atraso, modelos mais simples podem apresentar bom desempenho, embora com menor capacidade de capturar padrões mais complexos.